In [ ]:
import numpy as np
import pandas as pd
import os
import warnings
import random
import warnings
warnings.filterwarnings("ignore", category=UserWarning, module="xgboost")
warnings.filterwarnings('ignore')
import sys
from collections import defaultdict

import json
sys.path.append("../../src/experiment_util")
import experiment_utils
import experiment_constants

sys.path.append("../../src/irl")
import data_corruption_utils
import irl_utils
import reddit_processing_utils
import reddit_traj_construction
import irl_data_processing_utils



In [ ]:
with open("./config.json") as f:
    config = json.load(f)
    root_output_dir_path = config["root_output_dir_path"]
    deberta_weight_path = config["deberta_weight_path"]
    trolls_list_path = config["trolls_list_path"]
    processed_traj_path =  root_output_dir_path + "/users_tp_traj" 


In [ ]:
# get all processed users
processed_users = [os.path.splitext(f)[0] for f in os.listdir(processed_traj_path) if os.path.isfile(os.path.join(processed_traj_path, f))]
# get russian troll names
trollnames_df = pd.read_csv(trolls_list_path) 
trollnames_df['Username'] = trollnames_df['Username'].str.replace(r'^u/', '', regex=True)
troll_names = trollnames_df.Username.values

troll_users = list(set(troll_names) & set(processed_users))
non_troll_users = list(set(processed_users) - set(troll_names))

In [ ]:
all_user_df = pd.DataFrame(columns=[
    "user",
    "traj",
    "tp",
    "traj_count",
    "traj_total"
])

for u in list(set(processed_users)):
    filename = u + ".npz"
    full_path = os.path.join(processed_traj_path, filename)
    with np.load(full_path) as data:
        user = filename.split(".")[0]
        traj = data["traj"]
        tp = data["tp"]
        traj_count = data["traj_count"]
        if len(traj) == 0:
            print(user,"user has no traj")
            continue
        traj_total = np.sum(irl_utils.compute_state_count(traj.reshape(-1,2) , tp.shape[0], tp.shape[1]))
    
    new_row = {
        "user": user,
        "traj": traj,
        "tp": tp,
        "traj_count": traj_count,
        "traj_total":traj_total
    }
    all_user_df.loc[len(all_user_df)] = new_row

In [ ]:
all_user_df.loc[all_user_df.user.isin(troll_users),"russian"] = 1
all_user_df.loc[~all_user_df.user.isin(troll_users),"russian"] = 0
all_user_df.to_pickle(root_output_dir_path + "/all_user_traj_df.pkl")

In [ ]:
russian_df = all_user_df[all_user_df.user.isin(troll_users)].copy()
russian_df["russian"] = 1
non_russian_df = all_user_df[~all_user_df.user.isin(troll_users)].copy()
non_russian_df["russian"] = 0

In [ ]:
num_s = 12
num_a = 6
legality_matrix = experiment_constants.legal_transitions

num_runs = 25


sampled_matched_perturbed_df = pd.DataFrame(columns=[
    "user",
    "traj",
    "traj_window",
    "traj_perturbed",
    "tp_perturbed",
    "traj_counts_perturbed",
    "traj_total_perturbed",
    "run",
    "perturb_percent",
    "russian"
])

for dropped_percent in [0, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]:
    for run in range(num_runs):
        print(dropped_percent, "run:",run)
        seed = 100 + run
        df_russian_sampled, df_sampled = data_corruption_utils.matched_window(russian_df,non_russian_df)
        def perturb(x):
            return data_corruption_utils.perturb_traj(x, dropped_percent, num_s, num_a, legality_matrix)
        def tp(x):
            return irl_utils.compute_tp(x.reshape(-1, 2), num_s, num_a)
        def state_count(x):
            return irl_utils.compute_state_count(x.reshape(-1, 2), num_s, num_a)
        
        df_russian_sampled["traj_perturbed"] = df_russian_sampled["traj"].apply(perturb)
        df_sampled["traj_perturbed"] = df_sampled["traj"].apply(perturb)

        df_russian_sampled["tp_perturbed"] = df_russian_sampled["traj_perturbed"].apply(tp)
        df_sampled["tp_perturbed"] = df_sampled["traj_perturbed"].apply(tp)

        df_russian_sampled["traj_counts_perturbed"] = df_russian_sampled["traj_perturbed"].apply(state_count)
        df_sampled["traj_counts_perturbed"] = df_sampled["traj_perturbed"].apply(state_count)
        
        df_russian_sampled['traj_total_perturbed'] = df_russian_sampled['traj_counts_perturbed'].apply(np.sum)
        df_sampled['traj_total_perturbed'] = df_sampled['traj_counts_perturbed'].apply(np.sum)

        for i,row in df_russian_sampled.iterrows():
            new_row = {
                "user":row["user"],
                "traj":row["traj"],
                "traj_window":row["window"],
                "traj_perturbed":row["traj_perturbed"],
                "tp_perturbed":row["tp_perturbed"],
                "traj_counts_perturbed":row["traj_counts_perturbed"],
                "traj_total_perturbed":row["traj_total_perturbed"],
                "run":run,
                "perturb_percent":dropped_percent,
                "russian":row["russian"]
            }
            sampled_matched_perturbed_df.loc[len(sampled_matched_perturbed_df)] = new_row
            
        for i,row in df_sampled.iterrows():
            new_row = {
                "user":row["user"],
                "traj":row["traj"],
                "traj_window":row["window"],
                "traj_perturbed":row["traj_perturbed"],
                "tp_perturbed":row["tp_perturbed"],
                "traj_counts_perturbed":row["traj_counts_perturbed"],
                "traj_total_perturbed":row["traj_total_perturbed"],
                "run":run,
                "perturb_percent":dropped_percent,
                "russian":row["russian"]
            }
            sampled_matched_perturbed_df.loc[len(sampled_matched_perturbed_df)] = new_row

sampled_matched_perturbed_df["tp_perturbed_legalised"] = sampled_matched_perturbed_df["tp_perturbed"].apply(lambda x : irl_utils.legalise_tp(x,experiment_constants.legal_transitions) )

# save df
sampled_matched_perturbed_df.to_pickle(root_output_dir_path + "/sampled_matched_perturbed_df.pkl")